In [ ]:
import sys
from contextlib import redirect_stdout 
import io

sys.path.append('../')
sys.path.append('../../src')

In [ ]:
import pandas as pd

df = pd.read_parquet('../ssi_all_features.parquet')
df.shape

In [ ]:
before_delivery_med_features = df.loc[df['description'] == 'before_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
after_delivery_med_features = df.loc[df['description'] == 'after_delivery_medadmin_ingredient', 'feature_name'].unique().tolist()
or_timings_features = df.loc[df['description'] == 'or_timings', 'feature_name'].unique().tolist()
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='feature_value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
from preprocess import preprocess

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_preprocessed = preprocess(df_pivoted)

In [ ]:
df_pivoted_preprocessed.isna().sum().sort_values(ascending=False)

In [ ]:
from feature_filter import feature_filter

buf = io.StringIO()
with redirect_stdout(buf):
    df_pivoted_filtered = feature_filter(df_pivoted_preprocessed, skip_medadmin_ingredient=True)

del df_pivoted_preprocessed

In [ ]:
df_pivoted_filtered.isna().sum().sort_values(ascending=False)

In [ ]:
before_delivery_med_features = [x for x in before_delivery_med_features if x in df_pivoted_filtered.columns]
after_delivery_med_features = [x for x in after_delivery_med_features if x in df_pivoted_filtered.columns]
or_timings_features = [x for x in df_pivoted_filtered.columns if x.startswith('or_timings')]
len(before_delivery_med_features), len(after_delivery_med_features), len(or_timings_features)

In [ ]:
med_features = [x for x in df_pivoted_filtered.columns if 'medadmin_ingredient' in x]
len(med_features)  # this has more since it has _isna features in it

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

clf = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),
                   param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]},
                  #  param_grid= = {'min_samples_split': [500]} 
                   verbose=3,
                   cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
                   scoring=['f1', 'roc_auc'],
                   refit='roc_auc')

### 1 pcornet features only

In [ ]:
sys.path.insert(0, '../../')

from src.model_training import nested_cv_evaluation_feature_selection

In [ ]:
pcornet_features = [x for x in df_pivoted_filtered.columns \
                    if x not in or_timings_features and x != 'target_variable' and x != 'birthid' and x != 'mother_patid']
len(pcornet_features)

In [ ]:
# df_pivoted_filtered = df_pivoted_filtered.sample(frac=0.2)

In [ ]:
X = df_pivoted_filtered[pcornet_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_baseline = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                          outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
metrics_baseline

In [ ]:
import shap

def get_shap(clf, X, max_display=30):
    explainer = shap.TreeExplainer(clf, data=X,
                                   model_output='probability', feature_perturbation='interventional')
    shap_values = explainer(X, check_additivity=True)
    shap_values.values = shap_values.values[:,:,1]
    shap.plots.beeswarm(shap_values, max_display=max_display)
    return shap_values

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

### 2. med features only using start time

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

In [ ]:
X = df_pivoted_filtered[pcornet_features_subset]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_2 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

### 3. pcornet subset without start time and or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)
print(len(pcornet_features_subset))

In [ ]:
X = df_pivoted_filtered[pcornet_features_subset + or_timings_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_3 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

In [ ]:
metrics_baseline['test'] = 'pcornet everything'
metrics_2['test'] = 'pcornet_has_med + med_earliest_minute'
metrics_3['test'] = 'pcornet_has_med + or_timings'
all_metrics = pd.concat([metrics_baseline, metrics_2, metrics_3], axis=0)
all_metrics

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

all_metrics['outer_fold'] = all_metrics['outer_fold'].astype(str)
sns.barplot(all_metrics, y='outer_fold', x='roc_auc', hue='test')
plt.xlim([0.6, 0.75])

In [ ]:
sns.barplot(all_metrics, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])

### 4 only has med???

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

X = df_pivoted_filtered[pcornet_features_subset]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_4 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
metrics_baseline['test'] = 'pcornet everything'
metrics_2['test'] = 'pcornet_has_med + med_earliest_minute'
metrics_3['test'] = 'pcornet_has_med + or_timings'
metrics_4['test'] = 'pcornet_has_med'
all_metrics = pd.concat([metrics_baseline, metrics_2, metrics_3, metrics_4], axis=0)

sns.barplot(all_metrics, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])

### 5, earliest + or timings

In [ ]:
pcornet_features_subset = []
for x in pcornet_features:
    if x in med_features:
        if 'has_medadmin_ingredient' in x:
            pcornet_features_subset.append(x)
        if x.endswith('earliest_minute'):
            pcornet_features_subset.append(x)
    else:
        pcornet_features_subset.append(x)

X = df_pivoted_filtered[pcornet_features_subset + or_timings_features]
print(X.shape)
y = df_pivoted_filtered['target_variable']
mother_patid = df_pivoted_filtered['mother_patid']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 50, 70, 100, 200, 500]}

metrics_5 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid, groups=mother_patid,
                                                   outer_splits=5, inner_splits=5, random_state=range(10))

In [ ]:
metrics_baseline['test'] = 'pcornet everything'
metrics_2['test'] = 'pcornet_has_med + med_earliest_minute'
metrics_3['test'] = 'pcornet_has_med + or_timings'
metrics_4['test'] = 'pcornet_has_med'
metrics_5['test'] = 'pcornet_has_med + med_earluest_minute + or_timings'
all_metrics = pd.concat([metrics_baseline, metrics_2, metrics_3, metrics_4, metrics_5], axis=0)

sns.barplot(all_metrics, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])

In [ ]:
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', min_samples_split=100)
clf.fit(X, y)
shap_values = get_shap(clf, X, max_display=50)

In [ ]:
all_metrics.to_csv('importance_of_or_timings_for_paper.csv', index=False)

In [ ]:
all_metrics.groupby('test')['roc_auc'].describe()

In [ ]:
from scipy.stats import ttest_rel

a = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med', 'roc_auc']
b = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + or_timings', 'roc_auc']

ttest_rel(a, b)

In [ ]:
a = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med', 'roc_auc']
b = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute', 'roc_auc']

ttest_rel(a, b)

In [ ]:
a = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + or_timings', 'roc_auc']
b = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute', 'roc_auc']

ttest_rel(a, b)

### 6 figure out the interpretation

In [ ]:
import numpy as np

def return_null_col(df, col):
    df = df.copy()
    if 'medadmin_ingredient' in col:
        tmp = col.split('_medadmin_ingredient')
        isna_col = tmp[0] + '_has_medadmin_ingredient'
        df.loc[df[isna_col]==0, col] = np.nan
    else:
        df.loc[df[f"{col}_isna"]==1, col] = np.nan
    return df[col]

In [ ]:
import numpy as np

df_or_timings = df_pivoted_filtered[or_timings_features].copy()
cols_to_drop = []
for x in or_timings_features:
    if x.endswith('isna'):  continue
    df_or_timings[x] = return_null_col(df_or_timings, x)
    cols_to_drop.append(f"{x}_isna")
df_or_timings = df_or_timings.drop(columns=cols_to_drop)
df_or_timings.mean()

In [ ]:
col = '1815_UNKNOWN_before_delivery_medadmin_ingredient_earliest_minute'
val = return_null_col(df_pivoted_filtered, col)
(
    df_or_timings.corrwith(val, method='spearman')
                 .sort_values()
)

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt

tmp = pd.concat([-df_or_timings['or_timings_anes_start_datetime'], val], axis=1)

sns.scatterplot(tmp, x='or_timings_anes_start_datetime', y=col)
plt.xlim([-100, 3000])
plt.ylim([-100, 3000])
plt.xlabel("Anesthesia Start Time (minutes before delivery)")
plt.ylabel("Earliest time of bupivacaine (minutes before delivery)")

### 7. remake some plots for the main result

In [ ]:
all_metrics

In [ ]:
from scipy.stats import bootstrap
import numpy as np
from my_scorers import boostrap_ci_agg

all_metrics.groupby('test')['roc_auc'].apply(boostrap_ci_agg)

In [ ]:
tmp1 = all_metrics[all_metrics['test'] == 'pcornet_has_med']
tmp1['test'] = 'baseline'

tmp2 = all_metrics[all_metrics['test'] == 'pcornet_has_med + or_timings']
tmp2['test'] = 'baseline + or_timings'

tmp3 = all_metrics[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute']
tmp3['test'] = 'baseline + med_earliest_minute'

tmp4 = all_metrics[all_metrics['test'] == 'pcornet_has_med + med_earluest_minute + or_timings']
tmp4['test'] = 'all features'

tmp = pd.concat([tmp1, tmp2, tmp3, tmp4], axis=0)
tmp.columns.name = ''

sns.barplot(tmp, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])
plt.ylabel('')

In [ ]:
all_metrics.groupby('test')['ap'].apply(boostrap_ci_agg), all_metrics.groupby('test')['sens_80'].apply(boostrap_ci_agg)

In [ ]:
# compute delta AUC instead
roc_ref = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med', 'roc_auc']
roc = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + or_timings', 'roc_auc']
boostrap_ci_agg(roc - roc_ref)

In [ ]:
roc_ref = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med', 'roc_auc']
roc = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute', 'roc_auc']
boostrap_ci_agg(roc - roc_ref)

In [ ]:
roc_ref = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + or_timings', 'roc_auc']
roc = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute', 'roc_auc']
boostrap_ci_agg(roc - roc_ref)

In [ ]:
roc_ref = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med', 'roc_auc']
roc = all_metrics.loc[all_metrics['test'] == 'pcornet_has_med + med_earluest_minute + or_timings', 'roc_auc']
boostrap_ci_agg(roc - roc_ref)

In [ ]:
tmp1 = all_metrics[all_metrics['test'] == 'pcornet_has_med']
tmp1['test'] = 'Baseline EHR features'

tmp2 = all_metrics[all_metrics['test'] == 'pcornet_has_med + or_timings']
tmp2['test'] = 'Baseline EHR + \nintraoperative timing features'

tmp3 = all_metrics[all_metrics['test'] == 'pcornet_has_med + med_earliest_minute']
tmp3['test'] = 'Baseline EHR + \nmedication administration timing features'

tmp4 = all_metrics[all_metrics['test'] == 'pcornet_has_med + med_earluest_minute + or_timings']
tmp4['test'] = 'all features'

tmp = pd.concat([tmp1, tmp2, tmp3, tmp4], axis=0)
tmp.columns.name = ''

sns.barplot(tmp, x='roc_auc', y='test')
plt.xlim([0.60, 0.72])
plt.ylabel('')